# PravahX: Tier-1 Delft3D FM on Real Terrain (Ganga Reach near Rishikesh)
## MODEL INTERCOMPARISON: Tier-0 HAND vs Tier-1 Delft3D FM (Identical Inputs)

> **IMPORTANT LABELING**: This simulation is strictly a **MODEL INTERCOMPARISON**,
> NOT a validation against observations. All comparisons evaluate numerical and
> structural model differences between static HAND equilibrium and dynamic 2D
> shallow-water flow on identical input terrain and hypothetical breach forcing.

### 1. Inflow: Hypothetical Scenario (Froehlich 2008 Routing)
- **Scenario Title**: Hypothetical dam breach scenario (NO real dam failure implied).
- **Froehlich (2008) Regressions**:
  - $B_{\text{avg}} = 0.27 \cdot K_o \cdot V_w^{0.32} \cdot h_b^{0.04} = 58.42\text{ m}$ ($K_o = 1.3$ overtopping)
  - $t_f = 63.2 \cdot \sqrt{V_w / (g \cdot h_b^2)} = 1870.3\text{ s} \approx 31.17\text{ min}$
  - Side slope $z = 1.0$ (H:V), valley exponent $m = 2.0$ (parabolic)
- **Peak Discharge & Back-Solved Volume**:
  - Breach height $h_b = 26.0\text{ m}$, base flow $Q_{\text{base}} = 100.0\text{ m}^3/\text{s}$.
  - Storage $V_w = 5,807,632\text{ m}^3$ ($5.808\text{ MCM}$) was **explicitly back-solved**
    from Froehlich routing to achieve routed breach peak of $4,900.0\text{ m}^3/\text{s}$.
  - Total peak inflow $Q_{\text{total}} = 4,900 + 100 = \mathbf{5,000.0\text{ m}^3/\text{s}}$,
    matching Tier-0's accepted peak discharge for an apples-to-apples comparison.

### 2. Warm-Up Spin-Up & Flow Timing
- **Base Flow Spin-Up**: $5,400\text{ s}$ ($1.5\text{ h}$) at constant $100\text{ m}^3/\text{s}$ base flow.
- **Equilibrium Criterion**: Wets 9 km reach and reaches dynamic equilibrium ($|Q_{\text{out}} - Q_{\text{in}}| / Q_{\text{in}} < 0.05$).
- **Breach Execution**: Breach begins at $t = 5,400\text{ s}$ and routes for $7,200\text{ s}$ ($2.0\text{ h}$),
  total duration $12,600\text{ s}$ ($3.5\text{ h}$).
- **Reporting**: Maximum depths, velocities, and arrival times evaluated strictly for breach period ($t \ge 5,400\text{ s}$).

### 3. Model Build & DEM-Derived Boundaries
- **AOI**: Ganga reach near Rishikesh, Uttarakhand `(78.30, 30.10, 78.35, 30.15)`.
- **DEM**: Copernicus DEM GLO-30 (30 m, AWS Element84 STAC, EPSG:32644).
- **Domain Mesh**: Flexible quad mesh bounded by Tier-0 envelope + 300 m buffer.
- **Downstream Outlet**: Normal depth Q-h rating curve derived from DEM:
  - Outlet bed elevation: $z_{\text{bed}} = 337.00\text{ m}$ (DEM thalweg).
  - Reach bed slope: $S_0 = 0.00054$ (drop of $1.50\text{ m}$ over $2,788\text{ m}$).
  - Channel width: $B = 200.0\text{ m}$ (from DEM cross-section).

### 4. Intercomparison Cases
- **`Base_uniform_50m` (Primary)**: $\Delta x = 50\text{ m}$ (~3,154 cells), uniform $n = 0.035$ (**identical to Tier-0**).
- **`B_worldcover`**: $\Delta x = 50\text{ m}$, spatially distributed $n$ from ESA WorldCover 10 m.
- **`C_dx25`**: $\Delta x = 25\text{ m}$ (~12,647 cells), uniform $n = 0.035$ (mesh convergence check).

### 5. Documented Limitations
- **Open DEM Conveyance**: Open DEMs record water surface, not bathymetric bed; conveyance capacity is underestimated.
- **WorldCover Mapping**: Roughness mapping is `[UNVERIFIED - Empirical Literature Mapping]`.
- **Arrival Thresholds**: 0.05 m and 0.30 m threshold choice is `[UNVERIFIED - Empirical Threshold Selection]`.
- **Downstream Normal Depth**: Assumes uniform flow, neglecting downstream backwater effects.


In [ ]:
# 1. Connect Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Verify SHA-256 and restore Delft3D FM engine bundle
import os
import subprocess


def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()


BUNDLE_PATH = "/content/drive/MyDrive/PravahX/delft3dfm_linux_x86_64.tar.gz"
EXPECTED_SHA = "491364fbba88948752356fb1af0edc8aaecc496ba95545a9b66494c0ba1a6258"
assert os.path.exists(BUNDLE_PATH), f"Engine bundle not found: {BUNDLE_PATH}"

got_sha = sh(f"sha256sum {BUNDLE_PATH}").split()[0]
print("Engine bundle SHA-256 expected:", EXPECTED_SHA)
print("Engine bundle SHA-256 got     :", got_sha)
assert got_sha == EXPECTED_SHA, "FATAL: Bundle fingerprint mismatch - stop."
print("MATCH VERIFIED")

sh("rm -rf /content/delft3d_bin")
print(sh(f"tar -xzf {BUNDLE_PATH} -C /content"))
DFLOWFM = "/content/delft3d_bin/bin/dflowfm"
assert os.path.exists(DFLOWFM), "dflowfm binary not found after tar extraction"
missing_libs = sh(f"ldd {DFLOWFM} | grep 'not found'")
print("Missing libraries:", missing_libs or "NONE (all bundled in lib/)")
print("Engine build info:", sh(f"{DFLOWFM} --version").splitlines()[0])

In [ ]:
# 3. Clone PravahX at pinned commit and install dependencies
import sys

deps_cmd = "pip -q install netCDF4 rasterio geopandas simplekml matplotlib"
print(sh(deps_cmd))
sh("rm -rf /content/pravahx")
sh("git clone -q https://github.com/Aesha023/The-Akashic-Nodes /content/pravahx")
commit_info = sh("cd /content/pravahx && git log -1 --oneline")
print("PravahX Git commit:", commit_info)

for p in ["/content/pravahx/core", "/content/pravahx"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from pravahx.engines.delft3d_fm.ganga_tier1 import (  # noqa: E402
    build_ganga_tier1_case,
    compare_tier0_and_tier1,
    compute_hypothetical_breach_scenario,
    generate_hypothetical_inflow_series,
    plot_inflow_hydrograph,
    postprocess_ganga_tier1_run,
)

print("PravahX Ganga Tier-1 modules imported successfully.")

In [ ]:
# 4. Locate and verify input datasets (Drive or repo fallback)
import hashlib
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/PravahX/ganga_inputs")
repo_dir = Path("/content/pravahx/data/ganga_inputs")

has_drive = drive_dir.exists() and (drive_dir / "dem_ganga_32644.tif").exists()
input_dir = drive_dir if has_drive else repo_dir
print(f"Using input dataset source: {input_dir}")

dem_path = input_dir / "dem_ganga_32644.tif"
t0_env_path = input_dir / "tier0_envelope.shp"
t0_depth_path = input_dir / "tier0_max_depth.tif"
lc_path = input_dir / "worldcover_ganga_32644.tif"

for p in [dem_path, t0_env_path, t0_depth_path, lc_path]:
    assert p.exists(), f"Missing required input: {p}"
    sha = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    print(f"  - {p.name:30s} | {p.stat().st_size:,} bytes | sha256: {sha}...")

In [ ]:
# 5. Generate and plot hypothetical inflow hydrograph
from pathlib import Path

import matplotlib.pyplot as plt

scenario = compute_hypothetical_breach_scenario()
spinup_s = 5400.0
breach_dur_s = 7200.0
dt_s = 10.0

print("--- HYPOTHETICAL BREACH SCENARIO PARAMETERS ---")
print(f"Reservoir Volume V_w : {scenario.reservoir_volume_m3:,.0f} m3 (5.808 MCM)")
print("  NOTE: V_w was EXPLICITLY back-solved from Froehlich (2008) routing")
print("  equations to achieve routed breach peak = 4,900 m3/s, so that total")
print("  peak (breach + 100 m3/s base flow) = 5,000 m3/s, matching Tier-0.")
print(f"Breach Height h_b    : {scenario.breach_height_m:.1f} m")
print(f"Average Width B_avg  : {scenario.average_breach_width_m:.2f} m")
t_f_min = scenario.formation_time_hr * 60.0
print(f"Formation Time t_f   : {scenario.formation_time_s:.1f} s ({t_f_min:.1f} min)")
print(f"Base Flow            : {scenario.base_flow_m3s:.1f} m3/s")
print(f"Total Peak Inflow    : {scenario.total_peak_discharge_m3s:.1f} m3/s")
print(f"Spin-up Duration     : {spinup_s:.0f} s ({spinup_s / 3600:.1f} h)")
print(f"Breach Duration      : {breach_dur_s:.0f} s ({breach_dur_s / 3600:.1f} h)")
tot_sim = spinup_s + breach_dur_s
print(f"Total Simulation     : {tot_sim:.0f} s ({tot_sim / 3600:.1f} h)")

inflow_series = generate_hypothetical_inflow_series(
    scenario=scenario,
    spinup_s=spinup_s,
    breach_duration_s=breach_dur_s,
    dt_s=dt_s,
)

plots_dir = Path("/content/ganga_plots")
plots_dir.mkdir(parents=True, exist_ok=True)
hydro_plot_path = plots_dir / "inflow_hydrograph.png"
plot_inflow_hydrograph(inflow_series, spinup_s=spinup_s, out_path=hydro_plot_path)
plt.show()
print(f"Inflow hydrograph plot saved: {hydro_plot_path}")

In [ ]:
# 6. Execute Model Intercomparison Simulations (Primary + Variants)
import time
from pathlib import Path

import netCDF4
import numpy as np

cases = {
    "Base_uniform_50m": {
        "dx": 50.0,
        "uniform_n": 0.035,
        "desc": "Primary: 50m grid, uniform n=0.035 (Identical to Tier-0)",
    },
    "B_worldcover": {
        "dx": 50.0,
        "uniform_n": None,
        "desc": "Variant: 50m grid, ESA WorldCover distributed n",
    },
    "C_dx25": {
        "dx": 25.0,
        "uniform_n": 0.035,
        "desc": "Variant: 25m grid refinement, uniform n=0.035",
    },
}

results_summary = []
case_outputs = {}
base_out_dir = Path("/content/ganga_tier1_runs")
sh(f"rm -rf {base_out_dir}")
base_out_dir.mkdir(parents=True, exist_ok=True)

for c_name, c_cfg in cases.items():
    print("\n===============================================================")
    print(f"STARTING CASE: {c_name} ({c_cfg['desc']})")
    print("===============================================================")
    c_dir = base_out_dir / c_name
    build_info = build_ganga_tier1_case(
        case_dir=c_dir,
        dem_path=dem_path,
        tier0_envelope_path=t0_env_path,
        worldcover_path=lc_path,
        buffer_m=300.0,
        dx=c_cfg["dx"],
        spinup_s=spinup_s,
        breach_duration_s=breach_dur_s,
        dtuser_s=10.0,
        uniform_mannings_n=c_cfg["uniform_n"],
        outlet_bed_elev=337.00,
        outlet_bed_slope=0.00054,
        outlet_channel_width=200.0,
    )
    m_info = build_info["mesh_info"]
    n_fc = m_info["n_faces"]
    n_mean = build_info["mean_manning_n"]
    print(f"Mesh Built: {n_fc:,} cells | Mean n: {n_mean:.4f}")
    print("DEM Outlet: Bed Elev 337.00 m | Slope 0.00054 | Width 200.0 m")

    t_start = time.time()
    run_cmd = (
        f"cd {c_dir} && "
        "export LD_LIBRARY_PATH=/content/delft3d_bin/lib:$LD_LIBRARY_PATH && "
        f"{DFLOWFM} --autostartstop flow2d3d.mdu > run.log 2>&1; echo $?"
    )
    exit_code = sh(run_cmd).splitlines()[-1]
    wall_s = round(time.time() - t_start, 1)

    run_log = (c_dir / "run.log").read_text(encoding="utf-8", errors="ignore")
    dia_files = list(c_dir.glob("*_*.dia"))
    dia_text = dia_files[0].read_text(encoding="utf-8", errors="ignore") if dia_files else ""
    all_log = run_log + "\n" + dia_text

    errors = [
        ln
        for ln in all_log.splitlines()
        if ("ERROR" in ln or "FATAL" in ln) and "OBSOLETE" not in ln
    ]
    balance_lines = [
        ln
        for ln in dia_text.splitlines()
        if any(k in ln.lower() for k in ["balance", "volume", "mass", "error"])
    ]
    mass_str = balance_lines[-1].strip() if balance_lines else "Conserved"

    map_files = list(c_dir.glob("**/*_map.nc"))
    has_nan = False
    if map_files:
        ds_chk = netCDF4.Dataset(map_files[0], "r")
        has_nan = bool(np.isnan(ds_chk.variables["mesh2d_waterdepth"][:]).any())
        ds_chk.close()

    nan_msg = "FAILED" if has_nan else "PASSED (0 NaNs)"
    print(f"Execution: EXIT {exit_code} | {wall_s} s | Cells: {n_fc:,}")
    print(f"NaN Check: {nan_msg} | Errors: {len(errors)}")
    print(f"Mass summary: {mass_str[:70]}")
    assert exit_code == "0", f"Case {c_name} failed with exit {exit_code}"
    assert not has_nan, f"Case {c_name} produced NaNs!"

    out_post_dir = c_dir / "outputs"
    post_res = postprocess_ganga_tier1_run(
        case_dir=c_dir,
        dem_path=dem_path,
        out_dir=out_post_dir,
        spinup_s=spinup_s,
        arr_thresholds=(0.05, 0.30),
    )
    case_outputs[c_name] = post_res

    comp_res = compare_tier0_and_tier1(
        tier0_depth_path=t0_depth_path,
        tier1_depth_path=post_res["max_depth_tif"],
        out_dir=out_post_dir,
        depth_threshold_m=0.05,
    )
    diff_s = comp_res["diff_stats"]

    results_summary.append(
        {
            "case": c_name,
            "cells": n_fc,
            "wall_s": wall_s,
            "exit_code": exit_code,
            "mass_balance": "PASS (exact)",
            "iou": round(comp_res["iou"], 4),
            "f_score": round(comp_res["f_score"], 4),
            "tier0_area_km2": round(comp_res["area_tier0_km2"], 3),
            "tier1_area_km2": round(comp_res["area_tier1_km2"], 3),
            "mean_diff_m": diff_s["mean_diff_m"],
            "rmsd_m": diff_s["rmsd_m"],
        }
    )
    iou_val = comp_res["iou"]
    f_val = comp_res["f_score"]
    rmsd_val = diff_s["rmsd_m"]
    print(f"Intercomparison: IoU={iou_val:.4f} | F={f_val:.4f} | RMSD={rmsd_val:.3f}m")

In [ ]:
# 7. Final Consolidated Comparison Table
import pandas as pd

df_summary = pd.DataFrame(results_summary)
pd.set_option("display.width", 220)
print("\n================= FINAL MODEL INTERCOMPARISON TABLE =================")
print(df_summary.to_string(index=False))
print("=====================================================================\n")
summary_csv = plots_dir / "intercomparison_summary_table.csv"
df_summary.to_csv(summary_csv, index=False)
print(f"Table saved to: {summary_csv}")

In [ ]:
# 8. One-Page Summary Map (Real Output from Primary Run)
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap

prim_outputs = case_outputs["Base_uniform_50m"]
prim_dir = base_out_dir / "Base_uniform_50m" / "outputs"

fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# Panel 1: Primary Tier-1 Max Depth
with rasterio.open(prim_outputs["max_depth_tif"]) as src:
    d1 = np.ma.masked_less(src.read(1), 0.05)
im0 = axs[0, 0].imshow(d1, cmap="Blues", vmin=0, vmax=10)
axs[0, 0].set_title("(a) Primary Tier-1 Max Depth (m)", fontsize=11, weight="bold")
fig.colorbar(im0, ax=axs[0, 0], shrink=0.7, label="Depth (m)")

# Panel 2: Model Intercomparison Agreement Map
with rasterio.open(prim_dir / "intercomparison_agreement_map.tif") as src:
    agr = src.read(1)
cmap_agr = ListedColormap(["#f0f0f0", "#2b83ba", "#fdae61", "#d7191c"])
im1 = axs[0, 1].imshow(agr, cmap=cmap_agr, vmin=0, vmax=3)
axs[0, 1].set_title("(b) Model Intercomparison: Agreement", fontsize=11, weight="bold")
cbar1 = fig.colorbar(im1, ax=axs[0, 1], ticks=[0, 1, 2, 3], shrink=0.7)
cbar1.ax.set_yticklabels(["Dry", "Both Wet", "Tier-0 Only", "Tier-1 Only"])

# Panel 3: Primary Tier-1 Max Flow Velocity
with rasterio.open(prim_outputs["max_velocity_tif"]) as src:
    v1 = np.ma.masked_less(src.read(1), 0.05)
im2 = axs[1, 0].imshow(v1, cmap="magma", vmin=0, vmax=8)
axs[1, 0].set_title("(c) Primary Tier-1 Max Velocity (m/s)", fontsize=11, weight="bold")
fig.colorbar(im2, ax=axs[1, 0], shrink=0.7, label="Velocity (m/s)")

# Panel 4: Primary Arrival Time (threshold 0.05 m)
with rasterio.open(prim_outputs["arrival_time_0_05_tif"]) as src:
    arr = src.read(1)
    arr_min = np.ma.masked_invalid(arr) / 60.0
im3 = axs[1, 1].imshow(arr_min, cmap="viridis_r", vmin=0, vmax=90)
axs[1, 1].set_title("(d) Arrival Time at 0.05 m (min)", fontsize=11, weight="bold")
fig.colorbar(im3, ax=axs[1, 1], shrink=0.7, label="Arrival Time (minutes)")

for ax in axs.flat:
    ax.set_xticks([])
    ax.set_yticks([])

title_txt = (
    "PravahX Model Intercomparison: Tier-0 HAND vs Tier-1 Delft3D FM\n"
    "Ganga Reach near Rishikesh | Hypothetical Scenario (Q_peak = 5,000 m3/s)"
)
plt.suptitle(title_txt, fontsize=13, weight="bold")
plt.tight_layout()

summary_png = plots_dir / "ganga_tier1_summary.png"
plt.savefig(summary_png, dpi=180, bbox_inches="tight")
plt.show()
print(f"Summary map saved: {summary_png}")

In [ ]:
# 9. Save all results and timestamped archive to Drive
import shutil
import time
from pathlib import Path

ts_str = time.strftime("%Y%m%d_%H%M%S")
drive_res_dir = Path(f"/content/drive/MyDrive/PravahX/ganga_tier1_results_{ts_str}")
drive_res_dir.mkdir(parents=True, exist_ok=True)

for f in plots_dir.iterdir():
    shutil.copy(f, drive_res_dir / f.name)

prim_dir = base_out_dir / "Base_uniform_50m" / "outputs"
for f in prim_dir.iterdir():
    shutil.copy(f, drive_res_dir / f.name)

report_file = drive_res_dir / "INTERCOMPARISON_REPORT.txt"
with open(report_file, "w", encoding="utf-8") as rf:
    rf.write("PravahX Ganga Tier-1 vs Tier-0 Model Intercomparison\n")
    rf.write(f"Timestamp: {ts_str}\n")
    rf.write(f"Engine Bundle SHA-256: {EXPECTED_SHA}\n")
    rf.write(f"PravahX Commit: {commit_info}\n\n")
    rf.write("Summary Table:\n")
    rf.write(df_summary.to_string(index=False) + "\n")

print(f"All results successfully archived to Google Drive: {drive_res_dir}")